# Week 4 Lab 2 — Scalar and Field Surrogates

**Runtime:** CPU is sufficient; GPU optional.  
You need `combined_cavity_dataset.npz`, released by the instructor after Phase-1 quality control.


In [ ]:
from pathlib import Path
import csv, json, time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import tensorflow as tf

from cavity_project_utils import load_dataset, field_errors, save_predictions

SEED=690
np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
path=Path("combined_cavity_dataset.npz")
if not path.exists():
    raise FileNotFoundError("Upload combined_cavity_dataset.npz released by the instructor.")
data=load_dataset(path)
train=data["split"]=="train"; test=data["split"]=="test"
print("Re:",data["Re"]); print("split:",data["split"])
print("field shape:",data["u"].shape)


## 1. Dataset audit


In [ ]:
assert data["u"].shape==data["v"].shape==data["psi"].shape==data["omega"].shape
assert np.isfinite(data["u"]).all() and np.isfinite(data["v"]).all()
assert not np.any(train & test)
print("training Re",data["Re"][train])
print("blind Re",data["Re"][test])


## 2. Extract primary-vortex observables


In [ ]:
targets=[]
for psi in data["psi"]:
    j,i=np.unravel_index(np.argmin(psi),psi.shape)
    targets.append([psi[j,i],data["x"][i],data["y"][j]])
targets=np.asarray(targets)
names=["psi_min","vortex_x","vortex_y"]
for Re,q,s in zip(data["Re"],targets,data["split"]): print(Re,s,dict(zip(names,q)))


## 3. Linear-interpolation baseline


In [ ]:
def interpolate_rows(rstar, rtrain, ytrain):
    order=np.argsort(rtrain); rr=rtrain[order]; yy=ytrain[order]
    hi=np.searchsorted(rr,rstar)
    if hi==0 or hi==len(rr): raise ValueError("Interpolation requires bracketing cases")
    if rr[hi]==rstar: return yy[hi]
    lo=hi-1; w=(rstar-rr[lo])/(rr[hi]-rr[lo])
    return (1-w)*yy[lo]+w*yy[hi]

pred_interp=np.vstack([interpolate_rows(r,data["Re"][train],targets[train]) for r in data["Re"][test]])
print("Interpolation MAE",dict(zip(names,np.mean(np.abs(pred_interp-targets[test]),axis=0))))


## 4. Polynomial baseline


In [ ]:
poly=make_pipeline(PolynomialFeatures(2),StandardScaler(),LinearRegression())
poly.fit(data["Re"][train,None],targets[train])
pred_poly=poly.predict(data["Re"][test,None])
print("Polynomial MAE",dict(zip(names,np.mean(np.abs(pred_poly-targets[test]),axis=0))))


## 5. Compact DNN — repeat at three seeds


In [ ]:
rmin,rmax=data["Re"][train].min(),data["Re"][train].max()
qmean=targets[train].mean(0); qstd=targets[train].std(0)+1e-12
scale_r=lambda r: 2*(r-rmin)/(rmax-rmin)-1

def make_scalar_dnn(seed):
    tf.keras.utils.set_random_seed(seed)
    return tf.keras.Sequential([
        tf.keras.layers.Input((1,)),
        tf.keras.layers.Dense(32,activation="tanh"),
        tf.keras.layers.Dense(32,activation="tanh"),
        tf.keras.layers.Dense(3)])

seed_preds=[]
for seed in [11,22,33]:
    model=make_scalar_dnn(seed)
    model.compile(optimizer=tf.keras.optimizers.Adam(2e-3),loss="mse")
    cb=tf.keras.callbacks.EarlyStopping(monitor="loss",patience=150,restore_best_weights=True)
    model.fit(scale_r(data["Re"][train,None]),(targets[train]-qmean)/qstd,
              epochs=2500,verbose=0,callbacks=[cb])
    seed_preds.append(model.predict(scale_r(data["Re"][test,None]),verbose=0)*qstd+qmean)
pred_dnn=np.mean(seed_preds,axis=0); std_dnn=np.std(seed_preds,axis=0)
print("DNN MAE",dict(zip(names,np.mean(np.abs(pred_dnn-targets[test]),axis=0))))
print("Mean ensemble std",dict(zip(names,std_dnn.mean(0))))


In [ ]:
# Save the scalar comparison table
rows=[]
for method,pred in [("interpolation",pred_interp),("polynomial",pred_poly),("dnn_ensemble",pred_dnn)]:
    for j,Re in enumerate(data["Re"][test]):
        rows.append([method,float(Re),*targets[test][j],*pred[j]])
with open("week4_scalar_results.csv","w",newline="") as f:
    w=csv.writer(f); w.writerow(["method","Re","true_psi","true_xv","true_yv","pred_psi","pred_xv","pred_yv"]); w.writerows(rows)
print("saved week4_scalar_results.csv")


## 6. Full-field interpolation baseline


In [ ]:
def interpolate_field(rstar,field):
    return interpolate_rows(rstar,data["Re"][train],field[train])

field_rows=[]
for idx in np.where(test)[0]:
    up=interpolate_field(data["Re"][idx],data["u"])
    vp=interpolate_field(data["Re"][idx],data["v"])
    e=field_errors(data["u"][idx],data["v"][idx],up,vp)
    field_rows.append((data["Re"][idx],e))
field_rows


## 7. Coordinate DNN: (Re,x,y) to (u,v)


In [ ]:
Xg,Yg=np.meshgrid(data["x"],data["y"])
def point_samples(mask,stride=2):
    xx=Xg[::stride,::stride].ravel(); yy=Yg[::stride,::stride].ravel()
    X=[]; Q=[]
    for k in np.where(mask)[0]:
        rr=np.full_like(xx,scale_r(data["Re"][k]))
        X.append(np.c_[rr,2*xx-1,2*yy-1])
        Q.append(np.c_[data["u"][k,::stride,::stride].ravel(),data["v"][k,::stride,::stride].ravel()])
    return np.vstack(X).astype("float32"),np.vstack(Q).astype("float32")

Xtr,Qtr=point_samples(train,stride=2)
tf.keras.utils.set_random_seed(SEED)
coord=tf.keras.Sequential([tf.keras.layers.Input((3,)),
    tf.keras.layers.Dense(64,activation="tanh"),tf.keras.layers.Dense(64,activation="tanh"),
    tf.keras.layers.Dense(64,activation="tanh"),tf.keras.layers.Dense(2)])
coord.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss="mse")
coord.fit(Xtr,Qtr,epochs=500,batch_size=1024,verbose=0,
          callbacks=[tf.keras.callbacks.EarlyStopping(monitor="loss",patience=50,restore_best_weights=True)])
print("parameters",coord.count_params())


In [ ]:
def predict_coord(Re):
    xx=Xg.ravel(); yy=Yg.ravel(); rr=np.full_like(xx,scale_r(Re))
    X=np.c_[rr,2*xx-1,2*yy-1].astype("float32")
    q=coord.predict(X,verbose=0)
    n=len(data["x"]); return q[:,0].reshape(n,n),q[:,1].reshape(n,n)

idx=np.where(test)[0][0]; Re_star=data["Re"][idx]
up,vp=predict_coord(Re_star)
print("coordinate DNN",Re_star,field_errors(data["u"][idx],data["v"][idx],up,vp))
save_predictions("predictions_coordinate_dnn.npz",Re_star,data["x"],data["y"],
                 data["u"][idx],data["v"][idx],up,vp,"coordinate_dnn")


In [ ]:
err=np.hypot(up-data["u"][idx],vp-data["v"][idx])
fig,ax=plt.subplots(1,3,figsize=(14,4))
ax[0].streamplot(Xg,Yg,data["u"][idx],data["v"][idx],density=1.1); ax[0].set_title("CFD")
ax[1].streamplot(Xg,Yg,up,vp,density=1.1); ax[1].set_title("coordinate DNN")
im=ax[2].contourf(Xg,Yg,err,30); fig.colorbar(im,ax=ax[2]); ax[2].set_title("vector error")
for a in ax:a.set(xlabel="x/L",ylabel="y/L",aspect="equal")
plt.tight_layout();plt.show()


## Required analysis

1. Which scalar method is best on blind cases?  
2. Does ensemble spread correlate with actual DNN error?  
3. Why is the flattened grid not 46,475 independent flow cases?  
4. Compare coordinate-DNN field error with field interpolation.  
5. Report wall error, divergence, and both centerlines for one blind case.


## Article-output contract

<!-- MIE690A article-aligned validation v3 -->

**Role:** Figure 9: CFD/MLP blind-case fields and centerlines.

All manuscript-facing figures must be generated from retained numerical/model outputs through the documented notebook or shared helper, saved under `results/`, and accompanied by machine-readable metrics. Do not redraw curves by eye or substitute a screenshot for a solver-to-reference comparison. The complete ownership table and exact output filenames are in [`ARTICLE_FIGURE_MAP.md`](../../ARTICLE_FIGURE_MAP.md).
